In [31]:
from google.colab import drive
drive.mount('/content/drive')

import os
path_to_data = '/content/drive/MyDrive/CEMS'
print(os.listdir(path_to_data))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['EMSN194', 'EMSR352', 'EMSR416', 'EMSR466', 'EMSR339', 'EMSR468', 'EMSR417']


In [32]:
!pip install torch torchvision
!pip install rasterio  # for handling .tif images
!pip install albumentations  # for data augmentation


Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x78a5d46a3b00>
Traceback (most recent call last):
  File "/usr/lib/python3.12/weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):

KeyboardInterrupt: 


In [33]:
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

/tmp/ipython-input-2736003858.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/amp/grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [34]:
import os
import numpy as np
import torch
import rasterio
from torch.utils.data import Dataset

class SenForFloodEventDataset(Dataset):
    def __init__(self, root_dir, modalities, patch_size=512,
                 random_crop=True, transforms=None):
        self.root_dir = root_dir
        self.modalities = modalities
        self.patch_size = patch_size
        self.random_crop = random_crop
        self.transforms = transforms

        self.index = []  # list of (event_id, sample_id)

        # Go through all events
        events = sorted([f for f in os.listdir(root_dir)
                         if os.path.isdir(os.path.join(root_dir, f))])

        for event in events:
            before_path = os.path.join(root_dir, event, "s1_before_flood")
            tif_files = sorted([f for f in os.listdir(before_path)
                                if f.endswith(".tif")])

            # Extract sample prefixes: 00001 from 00001_s1_before_flood.tif
            sample_ids = [f.split("_")[0] for f in tif_files]

            for sid in sample_ids:
                self.index.append((event, sid))

    def __len__(self):
        return len(self.index)

    def load_raster(self, event, folder, sample_id):
        # find file starting with sample_id
        folder_path = os.path.join(self.root_dir, event, folder)
        file = [f for f in os.listdir(folder_path)
                if f.startswith(sample_id)]
        if len(file) != 1:
            raise RuntimeError(f"Missing file for {event} {folder} {sample_id}")

        path = os.path.join(folder_path, file[0])
        with rasterio.open(path) as src:
            return src.read()

    def crop(self, img, mask):
        _, H, W = img.shape
        ph = self.patch_size

        if self.random_crop:
            top = np.random.randint(0, H - ph)
            left = np.random.randint(0, W - ph)
        else:
            top = (H - ph) // 2
            left = (W - ph) // 2

        return (img[:, top:top+ph, left:left+ph],
                mask[:, top:top+ph, left:left+ph])

    def __getitem__(self, idx):
        event, sid = self.index[idx]

        # Load modalities
        arrays = []
        for folder in self.modalities:
            arrays.append(self.load_raster(event, folder, sid))
        x = np.concatenate(arrays, axis=0)

        # Load mask
        y = self.load_raster(event, "flood_mask", sid)
        y = (y > 0).astype(np.float32)

        # Crop
        x, y = self.crop(x, y)

        # Normalize
        x = x.astype(np.float32)
        for c in range(x.shape[0]):
            m, s = np.nanmean(x[c]), np.nanstd(x[c]) + 1e-6
            x[c] = (x[c] - m) / s

        return torch.tensor(x), torch.tensor(y)

In [35]:
import rasterio

path = "/content/drive/MyDrive/CEMS/EMSN194/s1_before_flood/000000_s1_before_flood.tif"

with rasterio.open(path) as src:
    print("Shape (C, H, W):", src.count, src.height, src.width)
    print("CRS:", src.crs)
    print("Dtype:", src.dtypes)

Shape (C, H, W): 4 512 512
CRS: EPSG:3857
Dtype: ('float32', 'float32', 'float32', 'float32')


In [36]:
from torch.utils.data import DataLoader

# Instantiate dataset
ds = SenForFloodEventDataset(
    root_dir="/content/drive/MyDrive/CEMS",   # <-- update if needed
    modalities=[
        "s1_before_flood",
        "s1_during_flood",
        "s2_before_flood",
        "s2_during_flood",
        "terrain",
        "LULC",
    ],
    patch_size=256,
    random_crop=False    # for easy reproducibility
)

print("Total samples =", len(ds))

# Show first 5 (event, sample_id) pairs
print("\nIndex preview:")
for i in range(len(ds)):
    print(i, ds.index[i])

x, y = ds[0]
print("\nLoaded sample shapes:")
print("x:", x.shape)   # (C, H, W)
print("y:", y.shape)   # (1, H, W)

# Check stats
print("\nx stats: mean =", x.mean().item(), ", std =", x.std().item())
print("y unique values:", torch.unique(y))


Total samples = 363

Index preview:
0 ('EMSN194', '000000')
1 ('EMSN194', '000001')
2 ('EMSN194', '000002')
3 ('EMSN194', '000003')
4 ('EMSN194', '000004')
5 ('EMSN194', '000005')
6 ('EMSN194', '000006')
7 ('EMSN194', '000007')
8 ('EMSN194', '000008')
9 ('EMSN194', '000009')
10 ('EMSN194', '000010')
11 ('EMSN194', '000011')
12 ('EMSN194', '000012')
13 ('EMSN194', '000013')
14 ('EMSN194', '000014')
15 ('EMSN194', '000015')
16 ('EMSN194', '000016')
17 ('EMSN194', '000017')
18 ('EMSN194', '000018')
19 ('EMSN194', '000019')
20 ('EMSN194', '000020')
21 ('EMSN194', '000021')
22 ('EMSN194', '000022')
23 ('EMSN194', '000023')
24 ('EMSN194', '000024')
25 ('EMSN194', '000025')
26 ('EMSN194', '000026')
27 ('EMSN194', '000027')
28 ('EMSN194', '000028')
29 ('EMSR339', '000000')
30 ('EMSR339', '000001')
31 ('EMSR339', '000002')
32 ('EMSR339', '000003')
33 ('EMSR339', '000004')
34 ('EMSR339', '000005')
35 ('EMSR339', '000006')
36 ('EMSR339', '000007')
37 ('EMSR339', '000008')
38 ('EMSR339', '000009')

In [37]:
from torch.utils.data import DataLoader

dataset = SenForFloodEventDataset(
    root_dir="/content/drive/MyDrive/CEMS",
    patch_size=256,
    random_crop=False,
    modalities=[
        "s1_before_flood",
        "s1_during_flood",
        "s2_before_flood",
        "s2_during_flood",
        "terrain",
        "LULC",
    ],
)

# Split dataset into training and validation
num_samples = len(dataset)
train_size = int(0.8 * num_samples)
val_size = num_samples - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

# Loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Test one batch
for x_batch, y_batch in train_loader:
    print("x_batch shape:", x_batch.shape)
    print("y_batch shape:", y_batch.shape)
    break

Training samples: 290, Validation samples: 73


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


x_batch shape: torch.Size([4, 27, 256, 256])
y_batch shape: torch.Size([4, 1, 256, 256])


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a5dc049c60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1628, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self._selector.poll(timeout)
    

In [38]:
print(len(train_loader.dataset))


290


In [39]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
from sklearn.metrics import jaccard_score
import numpy as np

In [40]:
# ----------------------
# Model building blocks
# ----------------------
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.double_conv(x)


In [41]:
class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x):
        return self.pool_conv(x)

In [42]:
class Up(nn.Module):
    def __init__(self, in_ch, out_ch, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_ch, out_ch)
        else:
            # convtranspose approach (less used here)
            self.up = nn.ConvTranspose2d(in_ch//2, in_ch//2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # pad if needed
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

In [43]:
class OutConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=1)
    def forward(self, x):
        return self.conv(x)

In [44]:
class UNetMultiModal(nn.Module):
    def __init__(self, in_channels=8, n_classes=1, bilinear=True):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes if n_classes>1 else 1)

    def forward(self, x):
        x1 = self.inc(x)        # (B,64,H,W)
        x2 = self.down1(x1)     # (B,128,H/2,W/2)
        x3 = self.down2(x2)     # (B,256,H/4,W/4)
        x4 = self.down3(x3)     # (B,512,H/8,W/8)
        x5 = self.down4(x4)     # (B,512 or 1024,H/16,W/16)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

In [45]:
# ----------------------
# Losses: Dice + Focal
# ----------------------
def dice_loss(pred, target, eps=1e-6):
    # pred: logits or probabilities. We will apply sigmoid inside.
    if pred.ndim == 4 and pred.shape[1] > 1:
        # multi-class improbable here; take class 1
        pred_p = torch.softmax(pred, dim=1)[:,1:2]
    else:
        pred_p = torch.sigmoid(pred)
    if target.ndim == 3:
        target = target.unsqueeze(1).float()
    else:
        target = target.float()
    inter = (pred_p * target).sum(dim=(2,3))
    denom = pred_p.sum(dim=(2,3)) + target.sum(dim=(2,3))
    dice = (2. * inter + eps) / (denom + eps)
    return 1 - dice.mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
        self.reduction = reduction
    def forward(self, logits, targets):
        if logits.ndim==4 and logits.shape[1]>1:
            logits = logits[:,1:2]
        if targets.ndim==3:
            targets = targets.unsqueeze(1).float()
        bce_loss = self.bce(logits, targets)
        p = torch.sigmoid(logits)
        pt = torch.where(targets == 1, p, 1 - p)
        w = (1 - pt) ** self.gamma
        loss = self.alpha * w * bce_loss
        return loss.mean() if self.reduction=='mean' else loss.sum()

class ComboLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.focal = FocalLoss(alpha=0.25, gamma=2.0)
    def forward(self, logits, targets):
        d = dice_loss(logits, targets)
        f = self.focal(logits, targets)
        return self.alpha * f + (1 - self.alpha) * d

In [46]:
# ----------------------
# Utilities: IoU per batch
# ----------------------
def batch_iou(pred_logits, target, threshold=0.5):
    # returns jaccard / IoU for binary
    if pred_logits.ndim==4 and pred_logits.shape[1]>1:
        pred = torch.argmax(pred_logits, dim=1)
    else:
        pred = (torch.sigmoid(pred_logits) > threshold).long().squeeze(1)
    if target.ndim==4:
        target = target.squeeze(1)
    pred_np = pred.detach().cpu().numpy().ravel()
    targ_np = target.detach().cpu().numpy().ravel()
    # handle degenerate case
    try:
        return jaccard_score(targ_np, pred_np, average='binary', zero_division=1)
    except Exception:
        return 0.0

In [47]:
# ----------------------
# Training config
# ----------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_channels = 27        # CHANGE if your channels differ
n_classes = 1
epochs = 10             # change as needed
learning_rate = 1e-4
weight_decay = 1e-5
checkpoint_dir = '/mnt/data/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

model = UNetMultiModal(in_channels=in_channels, n_classes=n_classes).to(device)
criterion = ComboLoss(alpha=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)
scaler = GradScaler()

print(f"Device: {device} — Model params (M): {sum(p.numel() for p in model.parameters())/1e6:.2f}")

Device: cpu — Model params (M): 13.41


/tmp/ipython-input-3145164648.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [48]:
# ----------------------
# TRAIN / VAL loop
# ----------------------
best_val_iou = 0.0
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    t0 = time.time()
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Train E{epoch+1}/{epochs}")
    for i, batch in pbar:
        # Expect batch = (images, masks)
        images, masks = batch[0].to(device), batch[1].to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(images)
            loss = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        pbar.set_postfix({'loss': running_loss/(i+1)})
    train_time = time.time() - t0

    # Validation
    model.eval()
    val_loss = 0.0
    val_iou = 0.0
    with torch.no_grad():
        for j, batch in enumerate(val_loader):
            images, masks = batch[0].to(device), batch[1].to(device)
            with autocast():
                logits = model(images)
                loss = criterion(logits, masks)
            val_loss += loss.item()
            val_iou += batch_iou(logits, masks)
    val_loss = val_loss / max(1, len(val_loader))
    val_iou = val_iou / max(1, len(val_loader))
    scheduler.step(val_iou)
    print(f"Epoch {epoch+1}/{epochs}  train_loss {running_loss/len(train_loader):.4f}  val_loss {val_loss:.4f}  val_iou {val_iou:.4f}  time {train_time:.1f}s")

    # checkpoint best
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        ckpt_path = os.path.join(checkpoint_dir, f'best_epoch{epoch+1}_iou{val_iou:.4f}.pth')
        torch.save({
            'epoch': epoch+1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler': scaler.state_dict(),
            'val_iou': val_iou
        }, ckpt_path)
        print(f"Saved best checkpoint: {ckpt_path}")

Train E1/10:   0%|          | 0/73 [00:00<?, ?it/s]WARNING:rasterio._env:CPLE_AppDefined in 000006_s2_before_flood.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
/tmp/ipython-input-2064616961.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.12/dist-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Train E1/10:   0%|          | 0/73 [00:12<?, ?it/s]


KeyboardInterrupt: 

In [49]:
# ----------------------
# Save final model
# ----------------------
final_path = os.path.join(checkpoint_dir, 'final_model.pth')
torch.save({'model_state_dict': model.state_dict()}, final_path)
print("Training finished. Final model saved to:", final_path)

Training finished. Final model saved to: /mnt/data/checkpoints/final_model.pth


In [ ]:
# ----------------------
# Inference helper
# ----------------------
def predict_mask(model, image_tensor, device=device, threshold=0.5):
    # image_tensor: (C,H,W) or (1,C,H,W)
    model.eval()
    with torch.no_grad():
        if image_tensor.ndim == 3:
            x = image_tensor.unsqueeze(0).to(device)
        else:
            x = image_tensor.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits)
        mask = (probs > threshold).long().squeeze(0).squeeze(0).cpu().numpy()
        probs_np = probs.squeeze(0).squeeze(0).cpu().numpy()
    return mask, probs_np

In [ ]:
# Visualization (optional)
def show_prediction(sample_image_np, pred_mask, gt_mask=None):
    # sample_image_np: H,W,C (e.g., RGB preview) - if >3 channels, pass [:,:,:3]
    import matplotlib.pyplot as plt
    fig_count = 3 if gt_mask is not None else 2
    fig, axes = plt.subplots(1, fig_count, figsize=(12, 4))
    if sample_image_np.shape[2] >= 3:
        axes[0].imshow(sample_image_np[...,:3])
    else:
        axes[0].imshow(sample_image_np[...,0], cmap='gray')
    axes[0].set_title('Input (preview)'); axes[0].axis('off')
    axes[1].imshow(pred_mask, cmap='gray'); axes[1].set_title('Predicted mask'); axes[1].axis('off')
    if gt_mask is not None:
        axes[2].imshow(gt_mask, cmap='gray'); axes[2].set_title('Ground truth'); axes[2].axis('off')
    plt.tight_layout()
    plt.show()


In [53]:
!cp /mnt/data/checkpoints/best_epoch10_iou0.7516.pth /content/drive/MyDrive/